# Hybrid Movie Recommender System (Content-Based + Collaborative Filtering)

This notebook builds, evaluates, and exports the machine learning models powering the Movie Recommender:
1. **Content-Based Filtering**: TMDB 5000 dataset (overview, genres, keywords, top 3 cast, director) vectorized with CountVectorizer & Cosine Similarity.
2. **Collaborative Filtering**: MovieLens dataset (user ratings) modeled using Matrix Factorization (**Surprise SVD** with latent item factor cosine similarity).
3. **Hybrid Recommender**: Dynamic alpha-weighted combination of content and collaborative similarities with cold-start fallback.
4. **Offline Evaluation**: Precision@10 & Recall@10 comparison table across active users.
5. **Manual Sanity Checks**: Qualitative verification on iconic benchmark movies.
6. **Artifact Export**: Exporting `movies.pkl`, `content_sim.pkl`, and `collab_sim.pkl` for the Flask backend.

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
import joblib
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Collaborative filtering with Surprise
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

print("Libraries successfully imported!")

## 1. Load Datasets and Link TMDB with MovieLens
We load `tmdb_5000_movies.csv`, `tmdb_5000_credits.csv`, `ratings.csv`, and `links.csv`.
We match TMDB `id` with MovieLens `links.csv` (`tmdbId`) and keep only movies present in **both** datasets.

In [ ]:
movies_raw = pd.read_csv("../data/tmdb_5000_movies.csv")
credits_raw = pd.read_csv("../data/tmdb_5000_credits.csv")
links_raw = pd.read_csv("../data/links.csv")
ratings_raw = pd.read_csv("../data/ratings.csv")

# Drop nulls in tmdbId and convert to int
links = links_raw.dropna(subset=["tmdbId"]).copy()
links["tmdbId"] = links["tmdbId"].astype(int)

# Merge TMDB movies + credits
merged_tmdb = movies_raw.merge(credits_raw[["movie_id", "cast", "crew"]], left_on="id", right_on="movie_id")

# Retain ALL TMDB movies to maximize catalog coverage
df = merged_tmdb.drop_duplicates(subset=["id"]).reset_index(drop=True)
df["year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year.fillna(0).astype(int)

print(f"Raw TMDB movies count: {len(movies_raw)}")
print(f"Raw MovieLens linked movies: {len(links)}")
print(f">>> Total movies retained in catalog: {len(df)} (Linked with MovieLens: {len(df[df['id'].isin(links['tmdbId'])])})")
df[["id", "title", "year", "vote_average"]].head()

## 2. Content-Based Feature Engineering
- Parse `genres`, `keywords`, `cast` (top 3), and `crew` (director) using `ast.literal_eval`
- Clean tokens by lowercasing and stripping internal spaces (e.g. `Sam Worthington` -> `samworthington`)
- Apply NLTK `PorterStemmer` to all tokens and concatenate into a single `tags` feature.

In [ ]:
stemmer = PorterStemmer()

def parse_names(json_str):
    try:
        items = ast.literal_eval(json_str)
        if isinstance(items, list):
            return [i["name"] for i in items if isinstance(i, dict) and "name" in i]
        return []
    except Exception:
        return []

def parse_top_cast(json_str, limit=3):
    try:
        items = ast.literal_eval(json_str)
        names = [i["name"] for i in items if isinstance(i, dict) and "name" in i]
        return names[:limit]
    except Exception:
        return []

def parse_director(json_str):
    try:
        items = ast.literal_eval(json_str)
        for i in items:
            if isinstance(i, dict) and i.get("job") == "Director":
                return [i.get("name")] if i.get("name") else []
        return []
    except Exception:
        return []

def clean_tokens(tokens):
    return [t.replace(" ", "").lower() for t in tokens if t]

df["parsed_genres"] = df["genres"].apply(parse_names)
df["parsed_keywords"] = df["keywords"].apply(parse_names)
df["parsed_cast"] = df["cast"].apply(parse_top_cast)
df["parsed_director"] = df["crew"].apply(parse_director).apply(lambda d: d[0] if d else "Unknown")

# Clean tokens for feature extraction
c_genres = df["parsed_genres"].apply(clean_tokens)
c_keywords = df["parsed_keywords"].apply(clean_tokens)
c_cast = df["parsed_cast"].apply(clean_tokens)
c_director = df["crew"].apply(parse_director).apply(clean_tokens)
c_overview = df["overview"].fillna("").apply(lambda text: [w.lower() for w in text.split()])

df["tags"] = c_overview + c_genres + c_keywords + c_cast + c_director
df["tags_string"] = df["tags"].apply(lambda tokens: " ".join([stemmer.stem(w) for w in tokens]))

print("Sample tags for ", df.loc[0, "title"], ":")
print(df.loc[0, "tags_string"][:250])

## 3. Vectorization and Content Similarity Matrix
We use `CountVectorizer` (vocabulary max 5000 features, English stop words) and compute the pairwise Cosine Similarity matrix (in `float32`).

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words="english")
vectors = cv.fit_transform(df["tags_string"]).toarray()
content_sim = cosine_similarity(vectors).astype(np.float32)
np.clip(content_sim, 0.0, 1.0, out=content_sim)

print(f"Feature matrix shape: {vectors.shape}")
print(f"Content similarity matrix shape: {content_sim.shape}, dtype: {content_sim.dtype}")

## 4. Collaborative Filtering using Surprise SVD
- Train SVD with an 80/20 train-test split, reporting RMSE and MAE.
- Fit full model to extract latent factor vectors (`qi`) for each movie.
- Compute item-item cosine similarity on the factor vectors.
- Normalize collaborative similarity matrix strictly to `[0, 1]`.
- Track cold-start movies (items with no collaborative ratings).

In [ ]:
merged_ratings = ratings_raw.merge(links[["movieId", "tmdbId"]], on="movieId")
valid_tmdb_ids = set(df["id"])
matched_ratings = merged_ratings[merged_ratings["tmdbId"].isin(valid_tmdb_ids)].copy()

print(f"Ratings count: {len(matched_ratings)} across {matched_ratings['userId'].nunique()} users")

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(matched_ratings[["userId", "tmdbId", "rating"]], reader)

# 80/20 train/test evaluation
trainset_split, testset_split = train_test_split(data, test_size=0.2, random_state=42)
eval_svd = SVD(n_factors=20, random_state=42)
eval_svd.fit(trainset_split)
preds = eval_svd.test(testset_split)
test_rmse = accuracy.rmse(preds)
test_mae = accuracy.mae(preds)
print(f">>> Surprise SVD Test RMSE: {test_rmse:.4f} | Test MAE: {test_mae:.4f}")

# Full training for factor extraction
full_trainset = data.build_full_trainset()
full_svd = SVD(n_factors=20, random_state=42)
full_svd.fit(full_trainset)

factors_list = []
has_collab_list = []
for tmdb_id in df["id"]:
    try:
        inner_id = full_trainset.to_inner_iid(tmdb_id)
        factors_list.append(full_svd.qi[inner_id])
        has_collab_list.append(True)
    except ValueError:
        factors_list.append(np.zeros(20, dtype=np.float32))
        has_collab_list.append(False)

df["has_collab"] = has_collab_list
item_factors = np.array(factors_list, dtype=np.float32)

collab_sim = cosine_similarity(item_factors).astype(np.float32)
np.nan_to_num(collab_sim, copy=False, nan=0.0)

# Min-max normalize to [0, 1]
collab_min = float(collab_sim.min())
collab_max = float(collab_sim.max())
collab_sim_norm = ((collab_sim - collab_min) / (collab_max - collab_min)).astype(np.float32)

print(f"Collaborative similarity matrix shape: {collab_sim_norm.shape}")
print(f"Movies with collaborative data: {sum(has_collab_list)} / {len(df)}")

## 5. Hybrid Recommender Function
Formula: `hybrid_score = alpha * content_sim + (1 - alpha) * collab_sim` (default $\alpha = 0.6$).
If a movie has no collaborative data (cold-start), we fall back directly to `content_sim`.

In [ ]:
def get_hybrid_scores(movie_idx, alpha=0.6):
    has_collab = df["has_collab"].values
    if not has_collab[movie_idx]:
        return content_sim[movie_idx].copy()
    scores = (alpha * content_sim[movie_idx]) + ((1.0 - alpha) * collab_sim_norm[movie_idx])
    no_collab = ~has_collab
    scores[no_collab] = content_sim[movie_idx][no_collab]
    return scores

def recommend_movies(title, n=10, alpha=0.6):
    matches = df[df["title"].str.lower() == title.lower()]
    if matches.empty:
        return f"Movie '{title}' not found."
    m_idx = matches.index[0]
    scores = get_hybrid_scores(m_idx, alpha=alpha)
    scores[m_idx] = -1.0  # Exclude target movie
    top_indices = np.argsort(scores)[::-1][:n]
    results = df.iloc[top_indices][["id", "title", "year", "vote_average"]].copy()
    results["score"] = scores[top_indices]
    return results

recommend_movies("The Dark Knight", n=5, alpha=0.6)

## 6. Offline Evaluation: Precision@10 & Recall@10 Comparison
We evaluate **Content-Only**, **Collaborative-Only**, and **Hybrid (alpha=0.6)** using active test users.

In [ ]:
k = 10
tmdb_to_idx = {tid: i for i, tid in enumerate(df["id"])}
high_ratings = matched_ratings[matched_ratings["rating"] >= 4.0]
user_counts = high_ratings.groupby("userId").size()
sample_users = user_counts[user_counts >= 15].index.tolist()[:100]

prec_content, rec_content = [], []
prec_collab, rec_collab = [], []
prec_hybrid, rec_hybrid = [], []

for user_id in sample_users:
    user_movies = high_ratings[high_ratings["userId"] == user_id]["tmdbId"].tolist()
    user_indices = [tmdb_to_idx[tid] for tid in user_movies if tid in tmdb_to_idx]
    if len(user_indices) < 8:
        continue
    n_seed = max(1, int(len(user_indices) * 0.7))
    seed = user_indices[:n_seed]
    holdout = set(user_indices[n_seed:])
    if not holdout:
        continue
    for mode, p_list, r_list in [
        ("content", prec_content, rec_content),
        ("collab", prec_collab, rec_collab),
        ("hybrid", prec_hybrid, rec_hybrid)
    ]:
        tot = np.zeros(len(df), dtype=np.float32)
        for s in seed:
            if mode == "content":
                tot += content_sim[s]
            elif mode == "collab":
                tot += collab_sim_norm[s]
            else:
                tot += get_hybrid_scores(s, alpha=0.6)
        tot[seed] = -1.0
        top_k = np.argsort(tot)[::-1][:k]
        hits = len(set(top_k).intersection(holdout))
        p_list.append(hits / k)
        r_list.append(hits / len(holdout))

eval_df = pd.DataFrame({
    "Model": ["Content-Only (alpha=1.0)", "Collaborative-Only (alpha=0.0)", "Hybrid (alpha=0.6)"],
    f"Precision@{k}": [np.mean(prec_content), np.mean(prec_collab), np.mean(prec_hybrid)],
    f"Recall@{k}": [np.mean(rec_content), np.mean(rec_collab), np.mean(rec_hybrid)]
})
eval_df

## 7. Qualitative Sanity Checks
Checking top 5 recommendations under Content, Collab, and Hybrid for benchmark titles.

In [ ]:
os.makedirs("../model", exist_ok=True)
movies_export = df[[
    "id", "title", "year", "parsed_genres", "parsed_cast",
    "parsed_director", "parsed_keywords", "overview",
    "vote_average", "vote_count", "popularity", "has_collab"
]].copy()
movies_export.rename(columns={
    "parsed_genres": "genres",
    "parsed_cast": "cast",
    "parsed_director": "director",
    "parsed_keywords": "keywords"
}, inplace=True)

joblib.dump(movies_export, "../model/movies.pkl")
joblib.dump(content_sim, "../model/content_sim.pkl")
joblib.dump(collab_sim_norm, "../model/collab_sim.pkl")
joblib.dump(cv, "../model/vectorizer.pkl")
joblib.dump(vectors, "../model/vectors.pkl")

print("Exported files:")
for fname in ["movies.pkl", "content_sim.pkl", "collab_sim.pkl", "vectorizer.pkl", "vectors.pkl"]:
    size_mb = os.path.getsize(f"../model/{fname}") / (1024 * 1024)
    print(f"  - {fname}: {size_mb:.2f} MB")

## 8. Export Model Artifacts
Saving `movies.pkl`, `content_sim.pkl`, and `collab_sim.pkl` into `../model/`.

In [ ]:
os.makedirs("../model", exist_ok=True)
movies_export = df[[
    "id", "title", "year", "parsed_genres", "parsed_cast",
    "parsed_director", "parsed_keywords", "overview",
    "vote_average", "vote_count", "popularity", "has_collab"
]].copy()
movies_export.rename(columns={
    "parsed_genres": "genres",
    "parsed_cast": "cast",
    "parsed_director": "director",
    "parsed_keywords": "keywords"
}, inplace=True)

joblib.dump(movies_export, "../model/movies.pkl")
joblib.dump(content_sim, "../model/content_sim.pkl")
joblib.dump(collab_sim_norm, "../model/collab_sim.pkl")

print("Exported files:")
for fname in ["movies.pkl", "content_sim.pkl", "collab_sim.pkl"]:
    size_mb = os.path.getsize(f"../model/{fname}") / (1024 * 1024)
    print(f"  - {fname}: {size_mb:.2f} MB")